# 01: Triage the Queue

You are the on-call security analyst. A critical alert triggered on one of your Linux hosts.
Find the alert, identify the host, and figure out what happened.

FalconPy is a Python SDK for the CrowdStrike Falcon API. Each "service class" (like `Alerts`,
`Hosts`, `SpotlightVulnerabilities`) handles authentication and gives you methods to query that
part of the platform. Every call returns a dictionary with `status_code` and `body`.

In [ ]:
import os

from falconpy import Alerts
from rich import print as rprint
from rich.table import Table

client_id = os.environ.get("FALCON_CLIENT_ID", "")
client_secret = os.environ.get("FALCON_CLIENT_SECRET", "")

alerts = Alerts(client_id=client_id, client_secret=client_secret)


def show(title, fields):
    table = Table(title=title)
    table.add_column("field", style="cyan")
    table.add_column("value")
    for k, v in fields.items():
        table.add_row(str(k), str(v))
    rprint(table)

Run the cell above to load the SDK, then run the cell below to confirm your credentials work.
If you see "authenticated" you are connected to your Falcon tenant.

In [ ]:
probe = alerts.query_alerts_v2(limit=1)

if probe["status_code"] != 200:
    print("auth failed:", probe["status_code"], probe["body"].get("errors"))
else:
    print("authenticated, alerts visible:", probe["body"]["meta"]["pagination"]["total"])

## Explore the endpoint

Two calls to learn the pattern:

1. `query_alerts_v2` searches for alerts and returns a list of IDs plus a count.
2. `get_alerts_v2` takes those IDs and returns the full alert records.

You filter results with FQL (Falcon Query Language). Filters look like `field:'value'` and you
join multiple filters with `+`. You can also `sort` by any field.

Run the cells below to see this in action.

In [ ]:
response = alerts.query_alerts_v2(
    limit=5,
    filter="product:'epp'",
    sort="created_timestamp.desc"
)

print("status_code:", response["status_code"])
print("epp alerts in the queue:", response["body"]["meta"]["pagination"]["total"])
print()
print("composite IDs returned (most recent first):")
for composite_id in response["body"]["resources"]:
    print(" ", composite_id)

In [ ]:
detail = alerts.get_alerts_v2(composite_ids=response["body"]["resources"])
alert = detail["body"]["resources"][0]

show("One alert record", {
    "product": alert.get("product"),
    "severity": alert.get("severity_name"),
    "tactic": alert.get("tactic"),
    "technique": alert.get("technique"),
    "filename": alert.get("filename"),
    "host": alert.get("host_names", ["?"])[0],
    "timestamp": alert.get("timestamp"),
})

Look at the fields on that alert record. The ones you can filter on include:

- `product` (what generated the alert, like `'epp'` for endpoint detections)
- `device.hostname` (which host)
- `severity_name` (like `'Critical'`, `'High'`, `'Medium'`, `'Low'`)
- `tactic` (MITRE ATT&CK tactic, like `'Privilege Escalation'` or `'Credential Access'`)

The `parent_details` and `grandparent_details` fields show the process tree, which you will use
after you find the case alert.

## Find the critical alert

Now write your own query. Filter for:
- `product:'epp'` (endpoint detections only)
- `severity_name:'Critical'` (critical severity)
- `device.hostname:'WS-DEV-01'` (the incident host)

Sort by `created_timestamp.desc` so the most recent alert comes first.

You saw all of these filters used in the explore cell above. Combine them with `+`.

In [ ]:
# TODO: query for critical epp alerts on WS-DEV-01, most recent first
# combine the three filters with + and add the sort
# hint: alerts.query_alerts_v2(filter="product:'epp'+severity_name:'...'+device.hostname:'...'", sort="...")
query = None  # replace this line with your query

if not query or query["status_code"] != 200:
    print("query failed or not filled in yet")
else:
    hits = query["body"]["resources"]
    print("critical alerts:", query["body"]["meta"]["pagination"]["total"])

In [ ]:
if not hits:
    print("no hits yet - fill in the TODO above first")
else:
    detail = alerts.get_alerts_v2(composite_ids=hits)["body"]["resources"]

    for alert in detail:
        print(alert["timestamp"], "|", alert["severity_name"], "|", alert.get("tactic"), "|", alert.get("filename"), "|", alert["host_names"][0])

    # the "Falcon Overwatch" entry is a human analyst escalation, not an automated detection
    # focus on the most recent Execution alert

In [ ]:
if not detail:
    print("no detail yet - run the cells above first")
else:
    trigger = detail[0]
    trigger_aid = trigger["agent_id"]

    show("Trigger", {
        "agent_id": trigger_aid,
        "host": trigger["host_names"][0],
        "tactic": trigger.get("tactic"),
        "filename": trigger.get("filename"),
    })

## Trace the attack chain

You have the host. Now pull the alerts that show how the attacker got in.

Every alert has a `parent_details` and `grandparent_details` field. These tell you what process
spawned the one that triggered the alert. Reading them is how you trace the chain backward from
the alert to the original entry point.

The cell below filters for "Credential Access" alerts on this host. Look at the `grandparent`
line in the output to see what process gave the attacker their initial shell.

In [ ]:
# initial access - what process gave the attacker a shell?
access_resp = alerts.query_alerts_v2(
    filter=f"agent_id:'{trigger_aid}'+product:'epp'+tactic:'Credential Access'"
)
access_alerts = alerts.get_alerts_v2(composite_ids=access_resp["body"]["resources"])["body"]["resources"]

print("Initial access:")
for alert in access_alerts:
    grandparent = (alert.get("grandparent_details") or {}).get("cmdline", "")
    parent = (alert.get("parent_details") or {}).get("cmdline", "")
    print(f"  user:        {alert.get('user_name')}")
    print(f"  grandparent: {grandparent}")
    print(f"  parent:      {parent}")
    print(f"  process:     {alert.get('filename')}")
    print()

The grandparent is `httpd`. The attacker came in through the web server, running as `daemon`.

Now look at how they escalated. Filter for Privilege Escalation on this host. Read the grandparent on those alerts to see what tool delivered the exploit.

In [ ]:
# privilege escalation - what delivered the exploit?
privesc_resp = alerts.query_alerts_v2(
    filter=f"agent_id:'{trigger_aid}'+product:'epp'+tactic:'Privilege Escalation'",
    sort="created_timestamp.desc"
)
privesc_alerts = alerts.get_alerts_v2(composite_ids=privesc_resp["body"]["resources"])["body"]["resources"]

print("Privilege escalation:")
for alert in privesc_alerts:
    grandparent = (alert.get("grandparent_details") or {}).get("cmdline", "")
    parent = (alert.get("parent_details") or {}).get("cmdline", "")
    print(f"  user:        {alert.get('user_name')}")
    print(f"  grandparent: {grandparent}")
    print(f"  parent:      {parent}")
    print(f"  process:     {alert.get('filename')}")
    print()

Look at the grandparent fields:

- One contains `.git/modules/.../post-checkout`. That is a git submodule hook. The attacker cloned a poisoned repository and the hook executed code on this host. The software responsible is the `git` package.

- The parent fields show `/tmp/pk.sh` and `/opt/pwnkit-poc/run.sh`. That is the PwnKit exploit targeting `pkexec`. The software responsible is `policykit-1`.

The chain: `httpd` gave the attacker a shell, a poisoned git submodule delivered the exploit, and pwnkit tried to escalate to root.

In notebook 02 you will search Spotlight for CVEs on `git` and `policykit-1`.

## Fallback

Run this only if the query above returned nothing.

In [ ]:
import json

try:
    with open("data/sample_responses/alerts_query.json") as handle:
        response = {"status_code": 200, "body": json.load(handle)}
except FileNotFoundError:
    print("sample file not found")
    response = None

if response:
    trigger = response["body"]["resources"][0]
    trigger_aid = trigger["agent_id"]
    show("Fallback", {"agent_id": trigger_aid, "host": trigger["host_names"][0]})

## Carry to 02

Copy the agent_id into notebook 02.

`trigger_aid = "<agent_id printed above>"`